In [1]:
import pandas as pd

df_fact = pd.read_csv("data/nyc_flights_fixed.csv")
df_fact.head()

,Unnamed: 0,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,carrier,flight,tailnum,origin,dest,air_time,distance,hour,minute,time_hour
0,0,2013,1,1,517,515,2,830,819,11,UA,1545,N14228,EWR,IAH,227,1400,5,15,2013-01-01T10:00:00Z
1,1,2013,1,1,533,529,4,850,830,20,UA,1714,N24211,LGA,IAH,227,1416,5,29,2013-01-01T10:00:00Z
2,2,2013,1,1,542,540,2,923,850,33,AA,1141,N619AA,JFK,MIA,160,1089,5,40,2013-01-01T10:00:00Z
3,3,2013,1,1,554,600,-6,812,837,-25,DL,461,N668DN,LGA,ATL,116,762,6,0,2013-01-01T11:00:00Z
4,4,2013,1,1,554,558,-4,740,728,12,UA,1696,N39463,EWR,ORD,150,719,5,58,2013-01-01T10:00:00Z


In [9]:
dim_airports = pd.read_csv("data/nyc_airports.csv")
dim_airlines = pd.read_csv("data/nyc_airlines.csv")
dim_planes = pd.read_csv("data/nyc_planes.csv")
dim_planes.head()

,tailnum,year,type,manufacturer,model,engines,seats,speed,engine
0,N10156,2004.0,Fixed wing multi engine,EMBRAER,EMB-145XR,2,55,NaN,Turbo-fan
1,N102UW,1998.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan
2,N103US,1999.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan
3,N104UW,1999.0,Fixed wing multi engine,AIRBUS INDUSTRIE,A320-214,2,182,NaN,Turbo-fan
4,N10575,2002.0,Fixed wing multi engine,EMBRAER,EMB-145LR,2,55,NaN,Turbo-fan


In [10]:
# chekc duplicates in faa
print(dim_airports.faa.duplicated().sum())
print(dim_planes.tailnum.duplicated().sum())
print(dim_airlines.carrier.duplicated().sum())

0
0
0


In [13]:
table = (
    df_fact
    .merge(dim_airports, left_on="origin", right_on="faa", how="left", suffixes=("", "_airport"))
    .merge(dim_airlines, left_on="carrier", right_on="carrier", how="left", suffixes=("", "_airline"))
    .merge(dim_planes, left_on="tailnum", right_on="tailnum", how="left", suffixes=("", "_plane"))
)

In [25]:
table

,Unnamed: 0,year,month,day,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,...,timezone_name,name_airline,year_plane,type,manufacturer,model,engines,seats,speed,engine
0,0,2013,1,1,517,515,2,830,819,11,...,America/New_York,United Air Lines Inc.,1999.0,Fixed wing multi engine,BOEING,737-824,2,149,NaN,Turbo-fan
1,1,2013,1,1,533,529,4,850,830,20,...,America/New_York,United Air Lines Inc.,1998.0,Fixed wing multi engine,BOEING,737-824,2,149,NaN,Turbo-fan
2,2,2013,1,1,542,540,2,923,850,33,...,America/New_York,American Airlines Inc.,1990.0,Fixed wing multi engine,BOEING,757-223,2,178,NaN,Turbo-fan
3,3,2013,1,1,554,600,-6,812,837,-25,...,America/New_York,Delta Air Lines Inc.,1991.0,Fixed wing multi engine,BOEING,757-232,2,178,NaN,Turbo-fan
4,4,2013,1,1,554,558,-4,740,728,12,...,America/New_York,United Air Lines Inc.,2012.0,Fixed wing multi engine,BOEING,737-924ER,2,191,NaN,Turbo-fan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277972,335736,2013,9,30,2240,2245,-5,2334,2351,-17,...,America/New_York,JetBlue Airways,2013.0,Fixed wing multi engine,EMBRAER,ERJ 190-100 IGW,2,20,NaN,Turbo-fan
277973,335737,2013,9,30,2240,2250,-10,2347,7,-20,...,America/New_York,JetBlue Airways,2007.0,Fixed wing multi engine,EMBRAER,ERJ 190-100 IGW,2,20,NaN,Turbo-fan
277974,335738,2013,9,30,2241,2246,-5,2345,1,-16,...,America/New_York,JetBlue Airways,2011.0,Fixed wing multi engine,EMBRAER,ERJ 190-100 IGW,2,20,NaN,Turbo-fan
277975,335739,2013,9,30,2307,2255,12,2359,2358,1,...,America/New_York,JetBlue Airways,2003.0,Fixed wing multi engine,AIRBUS,A320-232,2,200,NaN,Turbo-fan


In [26]:
# create date
table["date"] = pd.to_datetime(table[["year", "month", "day"]])

In [ ]:
table["date"]

0        2013-01-01
1        2013-01-01
2        2013-01-01
3        2013-01-01
4        2013-01-01
            ...    
277972   2013-09-30
277973   2013-09-30
277974   2013-09-30
277975   2013-09-30
277976   2013-09-30
Name: date, Length: 277977, dtype: datetime64[ns]

In [28]:
table.dest

0         IAH
1         IAH
2         MIA
3         ATL
4         ORD
         ... 
277972    SYR
277973    BUF
277974    ROC
277975    BOS
277976    BNA
Name: dest, Length: 277977, dtype: object

In [ ]:
# Calculate the rounded average of distinct destinations for each day.
avg_distinct_destinations = (
    table.groupby("date").agg({"dest":'nunique'})
    .dest.mean()
)

81.07671232876713

In [40]:
# Calculate the average number of destinations per day of the week.
# week day
week_day = lambda x: x.weekday()
table['weekday'] = table['date'].apply(week_day)
# avg
(
    table.groupby('weekday').agg({'dest':'nunique'})
)

,dest
weekday,
0,92
1,92
2,92
3,93
4,94
5,95
6,96


In [51]:
# Identify the month with the highest number of flights.
(
    table.groupby('month').agg({'tailnum':'count'})
    .sort_values('tailnum',ascending=False)
    .iloc[0]
)

tailnum    24290
Name: 8, dtype: int64